In [1]:
import os
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchaudio
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.notebook import tqdm

In [2]:
train_df = pd.read_csv('/kaggle/input/competitions/digitrecognition-ee708/train.csv')
print(train_df.shape)
print(train_df.head())

(37800, 2)
             id  label
0  train_000001      6
1  train_000002      6
2  train_000003      7
3  train_000004      7
4  train_000005      4


In [3]:
import shutil
if os.path.exists('/kaggle/working/mel_cache'):
    shutil.rmtree('/kaggle/working/mel_cache')
os.makedirs('/kaggle/working/mel_cache', exist_ok=True)

# ✅ Constants defined once, used everywhere
N_MELS = 128
N_FFT = 1024
HOP_LENGTH = 128
TARGET_LEN = 256

print("Caching mel spectrograms...")
for _, row in tqdm(train_df.iterrows(), total=len(train_df)):
    cache_path = f"/kaggle/working/mel_cache/{row['id']}.pt"
    if os.path.exists(cache_path):
        continue

    path = f"/kaggle/input/competitions/digitrecognition-ee708/train_audio/train_audio/{row['id']}.wav"
    waveform, sr = torchaudio.load(path)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    if sr != 16000:
        waveform = T.Resample(sr, 16000)(waveform)

    mel_transform = T.MelSpectrogram(
        sample_rate=16000, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH
    )
    mel = T.AmplitudeToDB()(mel_transform(waveform))
    mel = (mel - mel.mean()) / (mel.std() + 1e-6)

    if mel.shape[2] < TARGET_LEN:
        mel = torch.nn.functional.pad(mel, (0, TARGET_LEN - mel.shape[2]))
    else:
        mel = mel[:, :, :TARGET_LEN]

    torch.save(mel, cache_path)

print("Done caching!")

Caching mel spectrograms...


  0%|          | 0/37800 [00:00<?, ?it/s]

Done caching!


In [4]:
class DigitDataset(Dataset):
    def __init__(self, df, augment=False, cache_dir='/kaggle/working/mel_cache'):
        self.df = df.reset_index(drop=True)
        self.augment = augment
        self.cache_dir = cache_dir

    def __len__(self):
        return len(self.df)

    def add_noise(self, mel):
        noise = torch.randn_like(mel) * 0.05
        return mel + noise

    def random_volume(self, mel):
        gain = np.random.uniform(0.7, 1.3)
        return mel * gain

    def time_shift(self, mel):
        shift = np.random.randint(-20, 20)
        return torch.roll(mel, shift, dims=2)

    def spec_augment(self, mel, freq_mask=15, time_mask=40):
        mel = mel.clone()
        # Two frequency masks
        for _ in range(2):
            f = np.random.randint(1, freq_mask)
            f0 = np.random.randint(0, mel.shape[1] - f)
            mel[:, f0:f0+f, :] = 0
        # Two time masks
        for _ in range(2):
            t = np.random.randint(1, time_mask)
            t0 = np.random.randint(0, mel.shape[2] - t)
            mel[:, :, t0:t0+t] = 0
        return mel

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        mel = torch.load(f"{self.cache_dir}/{row['id']}.pt")

        if self.augment:
            mel = self.add_noise(mel)
            mel = self.random_volume(mel)
            mel = self.time_shift(mel)
            mel = self.spec_augment(mel)

        return mel, int(row['label'])

In [5]:
class DigitCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.2),
            # Block 2
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.2),
            # Block 3
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.3),
            # Block 4
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.3),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((4, 4)),
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {device}")

# ✅ Two separate dataset objects — no augment leak
train_df_reset = train_df.reset_index(drop=True)
val_size = int(0.1 * len(train_df_reset))
train_size = len(train_df_reset) - val_size

# Split indices first
indices = torch.randperm(len(train_df_reset)).tolist()
train_indices = indices[:train_size]
val_indices = indices[train_size:]

train_ds = DigitDataset(train_df_reset.iloc[train_indices], augment=True)
val_ds   = DigitDataset(train_df_reset.iloc[val_indices],   augment=False)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=128, shuffle=False, num_workers=4, pin_memory=True)

model     = DigitCNN().to(device)
optimizer = Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=50)
criterion = nn.CrossEntropyLoss()

best_val_acc = 0
for epoch in range(50):
    model.train()
    correct, total = 0, 0

    for mel, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/50"):
        mel, labels = mel.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(mel)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        correct += (out.argmax(1) == labels).sum().item()
        total += labels.size(0)

    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for mel, labels in val_loader:
            mel, labels = mel.to(device), labels.to(device)
            out = model(mel)
            val_correct += (out.argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    val_acc = val_correct / val_total
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), '/kaggle/working/best_model.pth')

    scheduler.step()
    print(f"Epoch {epoch+1:02d} | Train Acc: {correct/total:.3f} | Val Acc: {val_acc:.3f} | Best: {best_val_acc:.3f}")

Using: cuda


Epoch 1/50:   0%|          | 0/266 [00:00<?, ?it/s]

Epoch 01 | Train Acc: 0.577 | Val Acc: 0.888 | Best: 0.888


Epoch 2/50:   0%|          | 0/266 [00:00<?, ?it/s]

Epoch 02 | Train Acc: 0.794 | Val Acc: 0.938 | Best: 0.938


...

Epoch 48 | Train Acc: 0.974 | Val Acc: 0.992 | Best: 0.993


Epoch 49/50:   0%|          | 0/266 [00:00<?, ?it/s]

Epoch 49 | Train Acc: 0.975 | Val Acc: 0.992 | Best: 0.993


Epoch 50/50:   0%|          | 0/266 [00:00<?, ?it/s]

Epoch 50 | Train Acc: 0.974 | Val Acc: 0.992 | Best: 0.993


In [7]:
test_files = glob.glob('/kaggle/input/competitions/digitrecognition-ee708/test_audio/test_audio/*.wav')
test_ids = [os.path.basename(f).replace('.wav', '') for f in test_files]
test_df = pd.DataFrame({'id': test_ids})
print(f"Found {len(test_df)} test files")

Found 16200 test files


In [8]:
class TestDataset(Dataset):
    def __init__(self, df, audio_dir):
        self.df = df.reset_index(drop=True)
        self.audio_dir = audio_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = f"{self.audio_dir}/{row['id']}.wav"

        waveform, sr = torchaudio.load(path)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        if sr != 16000:
            waveform = T.Resample(sr, 16000)(waveform)

        mel_transform = T.MelSpectrogram(
            sample_rate=16000, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH
        )
        mel = T.AmplitudeToDB()(mel_transform(waveform))
        mel = (mel - mel.mean()) / (mel.std() + 1e-6)

        if mel.shape[2] < TARGET_LEN:
            mel = torch.nn.functional.pad(mel, (0, TARGET_LEN - mel.shape[2]))
        else:
            mel = mel[:, :, :TARGET_LEN]

        return mel, row['id']

In [9]:
model.load_state_dict(torch.load('/kaggle/working/best_model.pth'))
model.eval()

test_ds     = TestDataset(test_df, '/kaggle/input/competitions/digitrecognition-ee708/test_audio/test_audio')
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False, num_workers=4, pin_memory=True)

ids, preds = [], []
with torch.no_grad():
    for mel, batch_ids in tqdm(test_loader, desc="Predicting"):
        mel = mel.to(device)
        out = model(mel)
        preds.extend(out.argmax(1).cpu().numpy())
        ids.extend(batch_ids)

submission = pd.DataFrame({'id': ids, 'label': preds})
submission.to_csv('/kaggle/working/submission.csv', index=False)
print(f"Done! {len(submission)} predictions saved.")
print(submission['label'].value_counts().sort_index())

Predicting:   0%|          | 0/127 [00:00<?, ?it/s]

Done! 16200 predictions saved.
label
0    1579
1    1689
2    1626
3    1580
4    1595
5    1626
6    1591
7    1629
8    1644
9    1641
Name: count, dtype: int64
